# EDA Final — Online Shoppers Purchasing Intention

**Objetivo principal:** comprender la estructura, calidad y comportamiento del dataset antes de iniciar el proceso de modelado.

**Variable objetivo:** `Revenue`

## Diccionario de columnas

- **Administrative**: cantidad de páginas administrativas visitadas.
- **Administrative_Duration**: tiempo total (en segundos) dedicado a páginas administrativas.
- **Informational**: cantidad de páginas informativas visitadas.
- **Informational_Duration**: tiempo dedicado a páginas informativas.
- **ProductRelated**: cantidad de páginas de productos visitadas.
- **ProductRelated_Duration**: tiempo total observando productos.
- **BounceRates**: porcentaje de sesiones en las que el usuario abandonó el sitio casi inmediatamente, sin interacción significativa.
- **ExitRates**: probabilidad de que una página sea la última visitada antes de abandonar el sitio.
- **PageValues**: valor económico esperado de una página en función de su contribución a conversiones.
- **SpecialDay**: cercanía de la sesión a una fecha comercial especial.
- **Month**: mes de la sesión.
- **OperatingSystems**: sistema operativo del visitante.
- **Browser**: navegador del visitante.
- **Region**: región geográfica del visitante.
- **TrafficType**: fuente de tráfico.
- **VisitorType**: tipo de visitante (nuevo, recurrente, otro).
- **Weekend**: indica si la sesión ocurrió durante el fin de semana.
- **Revenue**: variable binaria que indica si la sesión terminó en una compra.

## 1. Configuración y carga de datos

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

In [ ]:
# Agregamos la raíz del proyecto al path para poder importar `src`
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import cargar_datos

df = cargar_datos()
print(f"Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas")

### Vista previa

In [ ]:
df.head(10)

## 2. Estructura y calidad de los datos

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
df.describe(include="object")

### Valores faltantes

In [ ]:
missing = pd.DataFrame({
    "Valores faltantes": df.isnull().sum(),
    "Porcentaje": round(df.isnull().mean() * 100, 2),
})

missing

### Duplicados

In [ ]:
duplicated_rows = df.duplicated().sum()
duplicated_percentage = duplicated_rows / len(df) * 100

print(f"Registros duplicados: {duplicated_rows:,}")
print(f"Porcentaje de duplicados: {duplicated_percentage:.2f}%")

In [ ]:
if duplicated_rows > 0:
    display(
        df[df.duplicated(keep=False)]
        .sort_values(by=df.columns.tolist())
        .head(10)
    )
else:
    print("No se detectaron registros duplicados exactos.")

> **Nota:** cada fila representa una sesión de navegación y el dataset no incluye un identificador único de usuario, por lo que estos duplicados podrían corresponder a sesiones distintas con características similares. Por este motivo no se eliminan automáticamente en esta etapa de EDA; la decisión de eliminarlos o no se toma en la etapa de preprocesamiento/feature engineering.

## 3. Variable objetivo: `Revenue`

In [ ]:
# True: la sesión terminó en una compra | False: no hubo compra
df["Revenue"].value_counts()

In [ ]:
round(df["Revenue"].value_counts(normalize=True) * 100, 2)

In [ ]:
plt.figure(figsize=(6, 5))
sns.countplot(data=df, x="Revenue")
plt.title("Distribución de la variable objetivo")
plt.xlabel("Compra realizada")
plt.ylabel("Cantidad")
plt.show()

La clase minoritaria (`Revenue = True`) representa una fracción reducida del dataset. Este desbalance deberá considerarse al elegir métricas y estrategias de entrenamiento del modelo.

## 4. Variables numéricas y categóricas

In [ ]:
numerical_columns = [
    "Administrative",
    "Administrative_Duration",
    "Informational",
    "Informational_Duration",
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
    "PageValues",
]

categorical_columns = [
    "Month",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType",
    "Weekend",
    "SpecialDay",
]

### Distribución de variables numéricas

In [ ]:
df[numerical_columns].hist(figsize=(18, 15), bins=30)
plt.tight_layout()
plt.show()

### Distribución de variables categóricas

In [ ]:
for column in categorical_columns:
    plt.figure(figsize=(8, 4))
    sns.countplot(data=df, x=column)
    plt.title(column)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 5. Análisis univariado y outliers

In [ ]:
for column in numerical_columns:
    plt.figure(figsize=(8, 2))
    sns.boxplot(x=df[column])
    plt.title(column)
    plt.tight_layout()
    plt.show()

In [ ]:
for column in numerical_columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    outliers = df[(df[column] < Q1 - 1.5 * IQR) | (df[column] > Q3 + 1.5 * IQR)]

    print(f"{column}: {len(outliers)} outliers ({len(outliers) / len(df) * 100:.2f}%)")

## 6. Análisis bivariado (vs `Revenue`)

In [ ]:
# Variables numéricas vs Revenue
for column in numerical_columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=df, x="Revenue", y=column)
    plt.title(f"{column} vs Revenue")
    plt.tight_layout()
    plt.show()

In [ ]:
# Variables categóricas vs Revenue
for column in categorical_columns:
    plt.figure(figsize=(8, 4))
    sns.countplot(data=df, x=column, hue="Revenue")
    plt.title(f"{column} vs Revenue")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 7. Correlaciones

In [ ]:
correlation = df[numerical_columns].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(correlation, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Matriz de correlación (variables numéricas)")
plt.show()

### Correlación de todas las variables (incluyendo categóricas codificadas) con `Revenue`

Codificamos las variables categóricas con `pd.get_dummies` únicamente para poder incluirlas en el análisis de correlación con la variable objetivo (no se usa para modelado en este notebook).

In [ ]:
df_encoded = pd.get_dummies(df, drop_first=True)
df_encoded["Revenue"] = df_encoded["Revenue"].astype(int)

correlation_target = (
    df_encoded.corr()["Revenue"]
    .drop("Revenue")
    .sort_values(key=abs, ascending=False)
)

correlation_target.head(15)

In [ ]:
plt.figure(figsize=(8, 6))
correlation_target.head(15).sort_values().plot(kind="barh", color="steelblue")
plt.title("Top 15 variables más correlacionadas con Revenue")
plt.xlabel("Correlación")
plt.tight_layout()
plt.show()

## 8. Conclusiones del EDA

A partir del Análisis Exploratorio de Datos realizado sobre el dataset **Online Shoppers Purchasing Intention**, se obtuvieron los siguientes hallazgos:

- El conjunto de datos está compuesto por **12.330 registros** y **18 variables**, proporcionando una base de información adecuada para el desarrollo de un modelo de clasificación orientado a predecir la intención de compra de los usuarios.
- No se identificaron **valores faltantes**, por lo que no será necesario realizar procesos de imputación durante la etapa de preprocesamiento.
- Se detectaron registros **duplicados** (~1% del dataset). Debido a que cada fila representa una sesión de navegación y no existe un identificador único de usuario, estos registros no se eliminan automáticamente en el EDA, ya que podrían corresponder a sesiones distintas con características similares.
- La variable objetivo **Revenue** presenta un **desbalance de clases** marcado (~85% no compra vs ~15% compra). Este comportamiento deberá considerarse durante el entrenamiento de los modelos y la selección de métricas de evaluación (por ejemplo, priorizar precision/recall/F1 sobre accuracy).
- El análisis de las variables numéricas mostró distribuciones fuertemente sesgadas y la presencia de valores atípicos en varias de ellas (destacan `PageValues`, `Informational` e `Informational_Duration`). Estos valores no se eliminan automáticamente, ya que pueden representar comportamientos reales de navegación y contener información relevante para la predicción.
- Las variables categóricas permitieron identificar diferentes perfiles de usuarios y características de las sesiones (mes, tipo de visitante, fin de semana), aportando información complementaria para el modelo.
- El análisis bivariado y las matrices de correlación (numérica y con variables categóricas codificadas) muestran que `PageValues`, `ExitRates`, `BounceRates` y `ProductRelated_Duration` están entre las variables más asociadas con `Revenue`, proporcionando una base sólida para la selección de características en las siguientes etapas del proyecto (`02_feature_engineering.ipynb`).